## Pytorch minimal workshop

### How to load data
First, import crucial dependencies

In [1]:

import numpy as np
import pandas as pd

import torch

# If you want to run PyTorch on GPU, you need to run a different torch setup 
# (now defined as additional dependency, to be activated by uv sync --extra cu126 
# This is highly dependent on your architecture!! Let me know if that is something of interest!

# print(torch.cuda.is_available())
# print(torch.cuda.get_device_name(0))





A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/mircodill/DSA104/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/mircodill/DSA104/.venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/mircodill/DSA104/.venv/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start
    self.io_loop.start(

Load a dataset using pandas. Pandas is still quite convenient for working with data, but ultimately, pytorch needs tensors (which are, just like DataFrames, built on top of numpy arrays). Any datasplits should also be done before transforming the dataframes into tensors.

In [2]:
from sklearn.datasets import load_diabetes
import pandas as pd

data = load_diabetes()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.DataFrame(data.target)

print(y.shape)
print(X.head())

(442, 1)
        age       sex       bmi        bp        s1        s2        s3  \
0  0.038076  0.050680  0.061696  0.021872 -0.044223 -0.034821 -0.043401   
1 -0.001882 -0.044642 -0.051474 -0.026328 -0.008449 -0.019163  0.074412   
2  0.085299  0.050680  0.044451 -0.005670 -0.045599 -0.034194 -0.032356   
3 -0.089063 -0.044642 -0.011595 -0.036656  0.012191  0.024991 -0.036038   
4  0.005383 -0.044642 -0.036385  0.021872  0.003935  0.015596  0.008142   

         s4        s5        s6  
0 -0.002592  0.019907 -0.017646  
1 -0.039493 -0.068332 -0.092204  
2 -0.002592  0.002861 -0.025930  
3  0.034309  0.022688 -0.009362  
4 -0.002592 -0.031988 -0.046641  


There are different ways to convert a dataframe into a tensor:

1) Directly from dataframe, via the `values` property.

MD: a tensor is just a specialized numpy array

In [3]:
X_tensor = torch.tensor(X.values) # only takes the values, and converts to tensor
X_tensor

tensor([[ 0.0381,  0.0507,  0.0617,  ..., -0.0026,  0.0199, -0.0176],
        [-0.0019, -0.0446, -0.0515,  ..., -0.0395, -0.0683, -0.0922],
        [ 0.0853,  0.0507,  0.0445,  ..., -0.0026,  0.0029, -0.0259],
        ...,
        [ 0.0417,  0.0507, -0.0159,  ..., -0.0111, -0.0469,  0.0155],
        [-0.0455, -0.0446,  0.0391,  ...,  0.0266,  0.0445, -0.0259],
        [-0.0455, -0.0446, -0.0730,  ..., -0.0395, -0.0042,  0.0031]])

2. Via `torch.tensor` but via numpy array. Likewise, numpy arrays can be directly converted into tensors.

In [4]:
X_tensor = torch.tensor(X.to_numpy(), dtype=torch.float32) # datatype conversion optional, first converts to numpy, then to tensor
X_tensor

tensor([[ 0.0381,  0.0507,  0.0617,  ..., -0.0026,  0.0199, -0.0176],
        [-0.0019, -0.0446, -0.0515,  ..., -0.0395, -0.0683, -0.0922],
        [ 0.0853,  0.0507,  0.0445,  ..., -0.0026,  0.0029, -0.0259],
        ...,
        [ 0.0417,  0.0507, -0.0159,  ..., -0.0111, -0.0469,  0.0155],
        [-0.0455, -0.0446,  0.0391,  ...,  0.0266,  0.0445, -0.0259],
        [-0.0455, -0.0446, -0.0730,  ..., -0.0395, -0.0042,  0.0031]])

3. Using `DataLoader` for large datasets (data larger than allocated memory)

MD: is not all data can be loaded into memory, you need a data laoder

In [5]:
from torch.utils.data import DataLoader, TensorDataset

X_tensor = torch.tensor(X.to_numpy())
y_tensor = torch.tensor(y.to_numpy())

dataset = TensorDataset(X_tensor, y_tensor)

dataloader = DataLoader(dataset, batch_size=2, shuffle=True) # selects random batches of 2 samples from the dataset, and shuffles the data at each epoch

for batch in dataloader:
    print(batch)

[tensor([[-0.0455,  0.0507,  0.1371, -0.0160,  0.0411,  0.0319, -0.0434,  0.0712,
          0.0710,  0.0486],
        [ 0.0199,  0.0507,  0.0143,  0.0632,  0.0149,  0.0203, -0.0471,  0.0343,
          0.0467,  0.0900]]), tensor([[233.],
        [297.]])]
[tensor([[-0.0564, -0.0446, -0.0116, -0.0332, -0.0470, -0.0477,  0.0045, -0.0395,
         -0.0080, -0.0881],
        [ 0.0708,  0.0507, -0.0170,  0.0219,  0.0438,  0.0563,  0.0376, -0.0026,
         -0.0702, -0.0176]]), tensor([[190.],
        [ 80.]])]
[tensor([[ 0.0018, -0.0446, -0.0709, -0.0229, -0.0016, -0.0010,  0.0266, -0.0395,
         -0.0225,  0.0072],
        [ 0.0308, -0.0446,  0.0056,  0.0115,  0.0782,  0.0779, -0.0434,  0.1081,
          0.0661,  0.0196]]), tensor([[ 49.],
        [249.]])]
[tensor([[-0.0782, -0.0446, -0.0407, -0.0814, -0.1006, -0.1128,  0.0229, -0.0764,
         -0.0203, -0.0508],
        [ 0.0344,  0.0507,  0.0283, -0.0332, -0.0456, -0.0098, -0.0508, -0.0026,
         -0.0595, -0.0218]]), tensor([[152.]

4. Use a Custom Dataset Class for more flexibility and cleaner code:

MD: defining a own class, which does exactly the same, but keeps rest of code clean. when initializing a new object as this class, it automatically converts to array / tensor

In [6]:

from torch.utils.data import Dataset, DataLoader

class MyDataFromDF(Dataset):
    def __init__(self, X, y):
        # features provided as Dataframes converted
        self.X = torch.tensor(    
            X.to_numpy(), 
            # dtype=torch.float32() # optionally include a datatype
            ) 
        
        # targets provided as Dataframes
        self.y = torch.tensor( 
            y.to_numpy(),
            # dtype=torch.float32() # optionally include a datatype
            )            

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]   # always return (features, target)


In [7]:
# Wrap custom dataset in Dataloader

dataset = MyDataFromDF(X, y)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

# for batch in loader:
#     print(batch)

In [8]:
# How to use the loader in training:
for batch_X, batch_y in loader:
    pred = model(batch_X)
    loss = criterion(pred, batch_y)
    ...


NameError: name 'model' is not defined

Datatypes need to be handled by torch (integer, float, Boolean) - compatible dtypes will be converted into a common format (check with `print(tensor.dtype)`). A datatype can be specified in (`dtype=torch.float32`) - see some examples above.

Strings, categorical values are incompatible and need to be encoded into numerical values.

## Build a NN for Regression
For convenience and better comparison, the Diabetes dataset as available in scikit-learn is used here again.

Train-test split on original data:

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Convert to tensors - you could also use any other method from above! Note that here there was no DataLoader used as wrapper, as the dataset is small.

MD: jsut everything needs to be a tensor

In [10]:
X_train = torch.tensor(X_train.to_numpy(), dtype=torch.float32)
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.float32)
X_test = torch.tensor(X_test.to_numpy(), dtype=torch.float32)
y_test = torch.tensor(y_test.to_numpy(), dtype=torch.float32)

A fully connected NN is defined as model class. Note the separation of the initialisation and the forward pass class functions (essentially separating the learning part from the activation functions).

In [118]:
import torch.nn as nn

class DiabetesNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(DiabetesNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)  # input layer with width = length of the feature set
        self.fc2 = nn.Linear(hidden_size, hidden_size)  # one hidden layer, try to add another one
        self.fc3 = nn.Linear(hidden_size, hidden_size)
        self.fc4 = nn.Linear(hidden_size, output_size)  # output layer
    
    # Good practice: activation functions specified in forward pass - separated from the learning parts
    def forward(self, x):            # forward pass
        x = torch.relu(self.fc1(x))  # ReLU activation function for input layer
        x = torch.relu(self.fc2(x))  # ReLU activation function for hidden layer
        x = torch.relu(self.fc3(x))  # ReLU activation function for the third layer
        # x = torch.relu(self.fc4(x))  # ReLU activation function for the fourth layer
        # no activation function for the output layer (because it is a regresion model!)
        return x

Define hyperparameters:

In [119]:
# Hyperparameters
input_size = X_train.shape[1]
hidden_size = 100  # here arbitrary, twice the input shape
output_size = 1
learning_rate = 0.005

# Number of training iterations
num_epochs = 2000

Introduce loss function (``criterion``) and gradient optimiser (``optimizer``):

MD: define / call the model

In [120]:
import torch.optim as optim
model = DiabetesNN(input_size, hidden_size, output_size)

criterion = nn.MSELoss()  # loss function defined; mean squared error is a good start for regression problems

optimizer = optim.Adam(model.parameters(), learning_rate) # gradient descent method based on average and squares of gradient

MD: from here the training really happens

In [121]:
for epoch in range(num_epochs):
    
    model.train()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    optimizer.zero_grad() # clear existing gradients
    loss.backward() # backpropagation of loss function to optimise gradient
    optimizer.step() # update parameters using SDG

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/100], Loss: {loss.item():.4f}')

Epoch [10/100], Loss: 29515.0781
Epoch [20/100], Loss: 28614.8574
Epoch [30/100], Loss: 26225.8945
Epoch [40/100], Loss: 21625.7031
Epoch [50/100], Loss: 15466.0371
Epoch [60/100], Loss: 11572.7539
Epoch [70/100], Loss: 11701.2471
Epoch [80/100], Loss: 11135.4443
Epoch [90/100], Loss: 11012.3379
Epoch [100/100], Loss: 10880.1475
Epoch [110/100], Loss: 10790.5059
Epoch [120/100], Loss: 10720.5156
Epoch [130/100], Loss: 10663.5566
Epoch [140/100], Loss: 10617.5020
Epoch [150/100], Loss: 10579.9795
Epoch [160/100], Loss: 10549.5801
Epoch [170/100], Loss: 10525.0088
Epoch [180/100], Loss: 10505.0811
Epoch [190/100], Loss: 10488.8711
Epoch [200/100], Loss: 10475.6182
Epoch [210/100], Loss: 10464.6973
Epoch [220/100], Loss: 10455.6182
Epoch [230/100], Loss: 10447.9932
Epoch [240/100], Loss: 10441.5078
Epoch [250/100], Loss: 10435.9180
Epoch [260/100], Loss: 10431.0215
Epoch [270/100], Loss: 10426.7148
Epoch [280/100], Loss: 10422.8604
Epoch [290/100], Loss: 10419.4922
Epoch [300/100], Loss: 

Evaluate the model on the test set to termine the generalisation for the regression model.

In [122]:
# Evaluate the model
mse_loss = nn.MSELoss()

model.eval()

with torch.no_grad():
    y_pred = model(X_test)
    mse = mse_loss(y_pred, y_test).item()
    print("MSE:", mse)


MSE: 9223.6455078125


In [123]:
# lowest reached so far: around 5000